In [1]:
import pandas as pd
import os
from datetime import datetime

# ==========================================
# 1. Configuration
# ==========================================
BASE_PATH = '..'
INPUT_DIR_RES = os.path.join(BASE_PATH, 'resultados')
INPUT_DIR_INC = os.path.join(BASE_PATH, 'includes')
OUTPUT_DIR = os.path.join(BASE_PATH, 'resultados', 'analises_mudanca')

# Arquivos de entrada
ARQUIVO_COMPARATIVO = 'comparativo_perfil_cenarios_qgis_2026_2035.csv'
ARQUIVO_CONSUMIDORES = 'Tabela_consumidores_Itapua_com_setor_comportamento_e_renda_2025_ajustado.csv'

def filtrar_matriculas_mudanca_comportamento(df, setor, cenario):
    """
    Filtra as matrículas que tiveram mudança de comportamento para um setor e cenário específicos
    """
    # Filtrar pelo setor e cenário específicos
    df_filtrado = df[(df['CD_SETOR'] == setor) & (df['NM_CENARIO'] == cenario)].copy()
    
    if len(df_filtrado) == 0:
        print(f"Nenhum registro encontrado para o setor {setor} e cenário {cenario}")
        return None
    
    # Identificar mudanças de comportamento
    df_filtrado['MUDOU_COMPORTAMENTO'] = df_filtrado['TP_COMPORTAMENTO'] != df_filtrado['TP_NOVO_COMPORTAMENTO']
    df_filtrado['TIPO_MUDANCA'] = df_filtrado.apply(
        lambda x: f"{x['TP_COMPORTAMENTO']} → {x['TP_NOVO_COMPORTAMENTO']}" if x['MUDOU_COMPORTAMENTO'] else "SEM_MUDANCA",
        axis=1
    )
    
    # Filtrar apenas quem mudou
    df_mudancas = df_filtrado[df_filtrado['MUDOU_COMPORTAMENTO']].copy()
    
    return df_mudancas

def cruzar_com_consumidores(df_mudancas, df_consumidores, setor, cenario):
    """
    Cruza os dados de mudanças com a tabela de consumidores
    """
    if df_mudancas is None or len(df_mudancas) == 0:
        print(f"Nenhuma mudança de comportamento encontrada para o setor {setor} no cenário {cenario}")
        return None
    
    # Selecionar colunas do comparativo que queremos manter
    colunas_comparativo = [
        'SK_MATRICULA', 'CD_SETOR', 'TP_COMPORTAMENTO', 'NN_MEDIA_CONSUMO',
        'NN_CONSUMO_DIARIO', 'NN_MORADORES_ANALISE', 'NM_CENARIO',
        'TP_NOVO_COMPORTAMENTO', 'NN_NOVA_MEDIA_CONSUMO', 'NN_NOVO_CONSUMO_DIARIO',
        'NN_NOVO_MORADORES_ANALISE', 'MUDOU_COMPORTAMENTO', 'TIPO_MUDANCA'
    ]
    
    # Garantir que todas as colunas existem
    colunas_comparativo_existentes = [col for col in colunas_comparativo if col in df_mudancas.columns]
    df_mudancas_selecionado = df_mudancas[colunas_comparativo_existentes]
    
    # Fazer o merge com a tabela de consumidores
    df_cruzado = pd.merge(
        df_mudancas_selecionado,
        df_consumidores,
        on='SK_MATRICULA',
        how='left'
    )
    
    return df_cruzado

def gerar_relatorio_mudancas(df_mudancas, setor, cenario, output_dir):
    """
    Gera arquivo apenas com as matrículas que mudaram (sem cruzar com consumidores)
    Opção 1 do script original
    """
    if df_mudancas is None or len(df_mudancas) == 0:
        print(f"Nenhuma mudança de comportamento encontrada para o setor {setor} no cenário {cenario}")
        return None
    
    # Criar diretório de saída se não existir
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Timestamp para o nome do arquivo
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. Arquivo completo com todas as colunas
    arquivo_completo = os.path.join(output_dir, f'matriculas_mudancas_{setor}_{cenario}_{timestamp}.csv')
    df_mudancas.to_csv(arquivo_completo, index=False, sep=',', encoding='utf-8-sig')
    
    # 2. Arquivo simplificado (apenas matrícula e informações de mudança)
    colunas_simplificado = ['SK_MATRICULA', 'CD_SETOR', 'TP_COMPORTAMENTO', 'TP_NOVO_COMPORTAMENTO', 
                           'TIPO_MUDANCA', 'NM_CENARIO']
    df_simplificado = df_mudancas[colunas_simplificado].copy()
    arquivo_simplificado = os.path.join(output_dir, f'matriculas_mudancas_{setor}_{cenario}_resumo_{timestamp}.csv')
    df_simplificado.to_csv(arquivo_simplificado, index=False, sep=',', encoding='utf-8-sig')
    
    # 3. Arquivo apenas com as SK_MATRICULA (lista simples)
    arquivo_matriculas = os.path.join(output_dir, f'matriculas_lista_{setor}_{cenario}_{timestamp}.txt')
    with open(arquivo_matriculas, 'w', encoding='utf-8') as f:
        for matricula in df_mudancas['SK_MATRICULA']:
            f.write(f"{matricula}\n")
    
    # Estatísticas
    print("\n" + "="*80)
    print(f"RELATÓRIO DE MUDANÇAS - Setor {setor} | Cenário {cenario}")
    print("="*80)
    print(f"Total de matrículas que mudaram: {len(df_mudancas)}")
    
    # Distribuição dos tipos de mudança
    print("\n--- Distribuição dos tipos de mudança ---")
    tipo_mudanca_counts = df_mudancas['TIPO_MUDANCA'].value_counts()
    for tipo, count in tipo_mudanca_counts.items():
        print(f"  {tipo}: {count} matrículas ({count/len(df_mudancas)*100:.1f}%)")
    
    print("\n" + "="*80)
    print(f"Arquivos gerados (apenas mudanças):")
    print(f"  1. Dados completos: {arquivo_completo}")
    print(f"  2. Dados resumidos: {arquivo_simplificado}")
    print(f"  3. Lista de matrículas: {arquivo_matriculas}")
    print("="*80)
    
    return df_mudancas

def gerar_relatorio_cruzado(df_cruzado, setor, cenario, output_dir):
    """
    Gera arquivo CSV com os dados cruzados (matrículas que mudaram + dados dos consumidores)
    Opção 2 do script
    """
    if df_cruzado is None or len(df_cruzado) == 0:
        print("Nenhum dado para salvar")
        return
    
    # Criar diretório de saída se não existir
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Timestamp para o nome do arquivo
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Nome do arquivo
    nome_arquivo = f'dados_cruzados_mudancas_{setor}_{cenario}_{timestamp}.csv'
    caminho_arquivo = os.path.join(output_dir, nome_arquivo)
    
    # Salvar CSV com vírgula como separador
    df_cruzado.to_csv(caminho_arquivo, index=False, sep=',', encoding='utf-8-sig')
    
    # Estatísticas
    print("\n" + "="*80)
    print(f"RELATÓRIO CRUZADO - Setor {setor} | Cenário {cenario}")
    print("="*80)
    print(f"Total de matrículas que mudaram: {len(df_cruzado)}")
    print(f"Total de colunas no arquivo: {len(df_cruzado.columns)}")
    
    # Distribuição dos tipos de mudança
    print("\n--- DISTRIBUIÇÃO DOS TIPOS DE MUDANÇA ---")
    tipo_mudanca_counts = df_cruzado['TIPO_MUDANCA'].value_counts()
    for tipo, count in tipo_mudanca_counts.items():
        print(f"  {tipo}: {count} matrículas ({count/len(df_cruzado)*100:.1f}%)")
    
    print("\n" + "="*80)
    print(f"Arquivo gerado com sucesso: {caminho_arquivo}")
    print(f"Total de registros: {len(df_cruzado)}")
    print(f"Total de colunas: {len(df_cruzado.columns)}")
    print("="*80)
    
    return caminho_arquivo

def listar_setores_disponiveis(df):
    """Lista todos os setores disponíveis no arquivo"""
    setores = sorted(df['CD_SETOR'].unique())
    print("\nSetores disponíveis:")
    print("-" * 40)
    for i, setor in enumerate(setores, 1):
        print(f"{i:3}. {setor}")
    return setores

def listar_cenarios_disponiveis(df):
    """Lista todos os cenários disponíveis no arquivo"""
    cenarios = sorted(df['NM_CENARIO'].unique())
    print("\nCenários disponíveis:")
    print("-" * 40)
    for i, cenario in enumerate(cenarios, 1):
        print(f"{i:3}. {cenario}")
    return cenarios

def filtrar_setor_cenario_direto(setor, cenario, opcao=2):
    """
    Função direta para filtrar sem interação
    
    Parâmetros:
    -----------
    setor : str
        Código do setor (ex: 'S014')
    cenario : str
        Nome do cenário (ex: 'CVIII')
    opcao : int
        1 - Apenas matrículas que mudaram
        2 - Dados cruzados com consumidores (padrão)
    
    Exemplo de uso:
    filtrar_setor_cenario_direto('S014', 'CVIII', opcao=2)
    """
    file_path = os.path.join(INPUT_DIR_RES, ARQUIVO_COMPARATIVO)
    consumidores_path = os.path.join(INPUT_DIR_INC, ARQUIVO_CONSUMIDORES)
    
    if not os.path.exists(file_path):
        print(f"Arquivo não encontrado: {file_path}")
        return None
    
    if opcao == 2 and not os.path.exists(consumidores_path):
        print(f"Arquivo de consumidores não encontrado: {consumidores_path}")
        return None
    
    df_comparativo = pd.read_csv(file_path, sep=',')
    
    if opcao == 2:
        df_consumidores = pd.read_csv(consumidores_path, sep=';')
    
    df_mudancas = filtrar_matriculas_mudanca_comportamento(df_comparativo, setor, cenario)
    
    if df_mudancas is None:
        return None
    
    if opcao == 1:
        gerar_relatorio_mudancas(df_mudancas, setor, cenario, OUTPUT_DIR)
        return df_mudancas
    else:
        df_cruzado = cruzar_com_consumidores(df_mudancas, df_consumidores, setor, cenario)
        if df_cruzado is not None:
            gerar_relatorio_cruzado(df_cruzado, setor, cenario, OUTPUT_DIR)
            return df_cruzado
    return None

# ==========================================
# MAIN - Escolha o modo de execução
# ==========================================

if __name__ == "__main__":
    # Modo 1: Interativo (pergunta qual setor, cenário e opção)
    #main_interativo()
    
    # Modo 2: Direto (descomente a linha abaixo e comente a linha acima)
    # filtrar_setor_cenario_direto('S014', 'CVIII', opcao=1)
    
    # Exemplo: cruzar com consumidores (opção 2)
    filtrar_setor_cenario_direto('S068', 'CVIII', opcao=2)


RELATÓRIO CRUZADO - Setor S068 | Cenário CVIII
Total de matrículas que mudaram: 61
Total de colunas no arquivo: 30

--- DISTRIBUIÇÃO DOS TIPOS DE MUDANÇA ---
  MODERADO → PERDULARIO: 25 matrículas (41.0%)
  AMBIENTALISTA → MODERADO: 19 matrículas (31.1%)
  AMBIENTALISTA → PERDULARIO: 17 matrículas (27.9%)

Arquivo gerado com sucesso: ..\resultados\analises_mudanca\dados_cruzados_mudancas_S068_CVIII_20260524_185411.csv
Total de registros: 61
Total de colunas: 30
